# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arjelmilan/flyrank-ai/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [23]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute("SET s3_endpoint='huggingface.co'")
con.execute(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN 'HT_TOKEN')")

TABLE = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

# quick sanity check
con.execute(f"""
    SELECT COUNT(*) AS n, MIN(report_date) AS min_d, MAX(report_date) AS max_d
    FROM {TABLE}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n,min_d,max_d
0,9841378,2026-03-01,2026-03-31


## 1. Unit of analysis + time window

Unit of analysis: One row represents one (report_date, client, content) observation of a content page's daily search performance.

Development window: I will develop the contract and feature logic using the 2026-03 warehouse partition.

Final evaluation: The 2026-06 sealed sample is reserved for final evaluation and will not be used to define labels, tune features, or make modeling decisions.

In [24]:


con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLE}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


## 2. Fields: feature / label / context / excluded

Label/proxy: not a raw column — we derive it ourselves as trend_direction = "down" by comparing gsc_clicks (or gsc_avg_position) in the current month vs. the prior month for the same content_hash_id. This is computed from gsc_clicks/gsc_avg_position across two time windows, so those columns can't also be used as features for the same window they define the label from.

Context (never a feature): client_hash_id, content_hash_id, month, report_date — identifiers/grouping only.

Excluded: client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available — these are availability flags, not signal; used for filtering, not modeling.

Features (knowable before the decision moment, i.e. from prior months only): gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, sessions_organic, etc. — all safe as long as they come from a month before the one being scored.

In [25]:
con.execute(f"""
    SELECT report_date, content_hash_id, gsc_clicks, gsc_impressions, gsc_avg_position
    FROM {TABLE}
    LIMIT 5
""").df()


,report_date,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position
0,2026-03-01,content_b7e512995f79d5a6,0,20,3.350000
1,2026-03-01,content_05597932fe4da067,0,1,0.000000
2,2026-03-01,content_7a105f548d9c6916,1,125,4.928000
3,2026-03-01,content_905aa32a0230694e,0,7,4.000000
4,2026-03-01,content_a3ea9792f793ec72,0,11,2.272727


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [26]:

con.execute(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {TABLE}
""").df()


,n_rows,n_content,min_date,max_date
0,9841378,331437,2026-03-01,2026-03-31


In [27]:


con.execute(f"""
    SELECT
      COUNT(*) AS n_total,
      SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS n_available
    FROM {TABLE}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_total,n_available
0,9841378,3611061.0


In [28]:

con.execute(f"""
    SELECT
      AVG(gsc_impressions) AS avg_impressions,
      AVG(gsc_clicks) AS avg_clicks,
      AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_excl_zero,
      AVG(ga4_sessions) AS avg_sessions,
      AVG(sessions_organic) AS avg_organic_sessions
    FROM {TABLE}
    WHERE gsc_data_available IS TRUE
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,avg_impressions,avg_clicks,avg_position_excl_zero,avg_sessions,avg_organic_sessions
0,77.721642,0.227587,16.575732,0.594785,0.277907


In [29]:
leak_check = con.execute(f"""
    SELECT content_hash_id,
           gsc_clicks AS feature_current_month_clicks,
           CASE WHEN gsc_clicks < AVG(gsc_clicks) OVER () THEN 'down' ELSE 'up' END AS trend_label_leaked
    FROM {TABLE}
    WHERE gsc_data_available IS TRUE
    LIMIT 5
""").df()
leak_check

,content_hash_id,feature_current_month_clicks,trend_label_leaked
0,content_b7e512995f79d5a6,0,down
1,content_05597932fe4da067,0,down
2,content_7a105f548d9c6916,1,up
3,content_905aa32a0230694e,0,down
4,content_a3ea9792f793ec72,0,down


## 4. Data limits

Only ~37% of rows have gsc_data_available = TRUE — the rest can't be used for GSC-based features/labels without filtering first. gsc_avg_position = 0 means "no data," not rank zero — must exclude before averaging (done above). History depth differs per client (per the data dictionary) — row counts aren't directly comparable across clients without checking dim_clients.gsc_data_start. Our label is a derived proxy (month-over-month click/position comparison), not a directly observed business outcome — claims should say "directional" or "decision-support," never "measured" or "observed" outcome. The _sample table (June 2026) is the natural outcome window for any trend label — must never be used to build or tune label logic, only to sanity-check query mechanics.

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.